# Myeloid Cell Cluster Validation & Re-annotation Pipeline

**Purpose**: Comprehensive validation of myeloid cluster annotations to identify:
- Authentic myeloid subclusters
- Doublets (e.g., Macrophage-Epithelial, Macrophage-Endothelial, Monocyte-B cell)
- Cross-lineage contamination
- Misclassified clusters

**Key Questions**:
1. Alveolar macrophages_c3: Real AM or AM-Endothelial doublet?
2. DC_c1: Real DC or DC-Epithelial doublet?
3. Intestinal macrophages_* (c0-c4): Real macrophages or contaminant?
4. Non-classical monocytes: Pure monocytes or Mono-B doublet?

**Validation Strategy**:
- Check CORE myeloid markers in ALL clusters
- Check CROSS-LINEAGE contamination markers
- Visualize on UMAP with multiple perspectives
- Generate dotplot/violin for quantitative assessment
- Provide confidence-scored re-annotation suggestions

**Output**:
- Comprehensive figures for each problem cluster
- Quantitative marker expression tables
- Re-annotation recommendations with confidence scores
- Updated metadata with validation flags

## 1. Configuration & Setup

In [ ]:
# ===== Import Libraries =====
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully")

In [ ]:
# ===== Configuration =====
# Paths
INPUT_PATH = "/home/h2048/data/py/0128/myeloid_analysis_unified/results/subcluster_unified_v2_20260128/adata_myeloid_subclustered_FINAL_v2_20260128.h5ad"
OUTPUT_DIR = Path("/home/h2048/data/py/0208/myeloid_validation")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Column names
CELLTYPE_COL = 'cell_type_scanvi_filt'  # Original fine-grained annotation
CLUSTER_COL = 'leiden'  # If you have leiden clusters
BATCH_KEY = 'sample'

# Scanpy settings
sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, facecolor='white', figsize=(8, 6))

print(f"Output directory: {OUTPUT_DIR}")
print(f"Configuration complete")

In [ ]:
# ===== Define Marker Gene Sets =====
# These are the core markers we'll use for validation

# CORE MYELOID MARKERS (should be present in ALL real myeloid cells)
CORE_MYELOID = {
    'Pan_Myeloid': ['LYZ', 'LST1', 'TYROBP', 'FCER1G', 'CTSS', 'CSF1R', 'AIF1'],
    'Macrophage_Core': ['C1QA', 'C1QB', 'C1QC', 'APOE', 'TREM2', 'MS4A7', 'MARCO'],
    'Monocyte_Core': ['S100A8', 'S100A9', 'S100A12', 'FCN1', 'VCAN', 'CD14'],
}

# SPECIFIC MYELOID SUBTYPES
MYELOID_SUBTYPES = {
    'Classical_Mono': ['S100A8', 'S100A9', 'FCN1', 'VCAN', 'CD14'],
    'Non_Classical_Mono': ['FCGR3A', 'LST1', 'LILRB2', 'IFITM2', 'IFITM3'],
    'Neutrophil': ['CXCR1', 'CXCR2', 'FCGR3B', 'CSF3R', 'CXCL8'],
    'Alveolar_Mac': ['MARCO', 'APOE', 'APOC1', 'MSR1', 'LPL', 'FABP4'],
    'Resident_Mac': ['FOLR2', 'LYVE1', 'MRC1', 'CD163', 'MERTK', 'SELENOP'],
    'Inflammatory_Mac': ['IL1B', 'CXCL2', 'CCL3', 'CCL4', 'TNF'],
    'cDC1': ['XCR1', 'CLEC9A', 'BATF3', 'IRF8'],
    'cDC2': ['CD1C', 'CD1E', 'FCER1A', 'CLEC10A'],
    'pDC': ['LILRA4', 'IL3RA', 'TCF4', 'CLEC4C'],
    'Mast': ['KIT', 'TPSAB1', 'TPSB2', 'CPA3', 'HDC'],
}

# CONTAMINATION MARKERS (should NOT be in myeloid cells)
CONTAMINATION_MARKERS = {
    'Epithelial': ['EPCAM', 'KRT19', 'KRT18', 'KRT8', 'CDH1'],
    'Airway_Epithelial': ['BPIFA1', 'SCGB1A1', 'MUC5AC', 'FOXJ1'],
    'Alveolar_Epithelial': ['SFTPA1', 'SFTPA2', 'SFTPB', 'SFTPC', 'SFTPD'],
    'B_Cell': ['MS4A1', 'CD79A', 'CD79B', 'IGHM', 'IGKC', 'IGLC1'],
    'T_Cell': ['CD3D', 'CD3E', 'CD4', 'CD8A', 'IL7R'],
    'Endothelial': ['PECAM1', 'VWF', 'KDR', 'EMCN', 'CDH5'],
    'Fibroblast': ['COL1A1', 'COL3A1', 'DCN', 'LUM'],
    'Pericyte': ['RGS5', 'PDGFRB', 'ACTA2'],
}

# PROBLEM CLUSTERS (from your analysis)
HIGH_RISK_CLUSTERS = {
    'Alveolar macrophages_c3': 'suspected_AM_Endothelial_doublet',
    'DC_c1': 'suspected_DC_Epithelial_doublet',
    'Intestinal macrophages_c0': 'suspected_contaminant',
    'Intestinal macrophages_c1': 'suspected_contaminant',
    'Intestinal macrophages_c2': 'suspected_contaminant',
    'Intestinal macrophages_c3': 'suspected_contaminant',
    'Intestinal macrophages_c4': 'suspected_contaminant',
    'Non-classical monocytes_c0': 'suspected_Mono_B_doublet',
    'Non-classical monocytes_c1': 'suspected_Mono_B_doublet',
}

print("Marker gene sets defined:")
print(f"  Core myeloid markers: {sum(len(v) for v in CORE_MYELOID.values())} genes")
print(f"  Myeloid subtype markers: {sum(len(v) for v in MYELOID_SUBTYPES.values())} genes")
print(f"  Contamination markers: {sum(len(v) for v in CONTAMINATION_MARKERS.values())} genes")
print(f"  High-risk clusters to validate: {len(HIGH_RISK_CLUSTERS)}")

## 2. Load Data

In [ ]:
# ===== Load Dataset =====
print("Loading myeloid dataset...")
adata = sc.read_h5ad(INPUT_PATH)

print(f"Dataset loaded: {adata.n_obs:,} cells × {adata.n_vars} genes")
print(f"\nData structure:")
print(f"  Layers: {list(adata.layers.keys())}")
print(f"  Obsm: {list(adata.obsm.keys())}")
print(f"  .raw present: {adata.raw is not None}")
if adata.raw is not None:
    print(f"  .raw genes: {adata.raw.n_vars}")

In [ ]:
# ===== Check Available Columns =====
print(f"\nKey metadata columns:")
if CELLTYPE_COL in adata.obs.columns:
    print(f"  ✓ Cell type column: {CELLTYPE_COL}")
    celltype_counts = adata.obs[CELLTYPE_COL].value_counts()
    print(f"    Unique cell types: {len(celltype_counts)}")
else:
    print(f"  ❌ Cell type column '{CELLTYPE_COL}' not found!")
    print(f"  Available columns: {list(adata.obs.columns)}")

if BATCH_KEY in adata.obs.columns:
    print(f"  ✓ Batch key: {BATCH_KEY}")
    print(f"    Unique batches: {adata.obs[BATCH_KEY].nunique()}")
else:
    print(f"  ⚠️  Batch key '{BATCH_KEY}' not found")

In [ ]:
# ===== Display Cell Type Distribution =====
if CELLTYPE_COL in adata.obs.columns:
    print("\nCell type distribution:")
    print("=" * 80)
    celltype_counts = adata.obs[CELLTYPE_COL].value_counts()
    
    for ct, count in celltype_counts.items():
        pct = count / adata.n_obs * 100
        risk_flag = "⚠️" if ct in HIGH_RISK_CLUSTERS else "✓"
        print(f"  {risk_flag} {ct}: {count:,} cells ({pct:.1f}%)")
    
    print("=" * 80)
    print(f"\n⚠️  High-risk clusters to validate: {len([ct for ct in celltype_counts.index if ct in HIGH_RISK_CLUSTERS])}")

## 3. Prepare Visualization Data

In [ ]:
# ===== Ensure log1p layer for visualization =====
print("Preparing data for visualization...")

if 'log1p' in adata.layers:
    print("  ✓ Using existing log1p layer")
    # Set X to log1p for visualization
    adata.X = adata.layers['log1p']
elif 'counts' in adata.layers:
    print("  ⚙️  Generating log1p layer from counts")
    # Create temporary normalized object for visualization
    adata.layers['log1p'] = adata.layers['counts'].copy()
    sc.pp.normalize_total(adata, target_sum=1e4, layer='log1p')
    sc.pp.log1p(adata, layer='log1p')
    adata.X = adata.layers['log1p']
else:
    print("  ⚠️  Using .X as-is (assuming already normalized)")

print("  ✓ Data ready for visualization")

In [ ]:
# ===== Check Marker Gene Availability =====
print("\nChecking marker gene availability...")

def check_markers_in_data(marker_dict, adata_var_names):
    """Check which markers are present in the dataset"""
    results = {}
    for category, genes in marker_dict.items():
        present = [g for g in genes if g in adata_var_names]
        missing = [g for g in genes if g not in adata_var_names]
        results[category] = {
            'present': present,
            'missing': missing,
            'coverage': len(present) / len(genes) * 100 if genes else 0
        }
    return results

# Get gene names (use .raw if available for full gene set)
gene_names = adata.raw.var_names if adata.raw is not None else adata.var_names

# Check all marker sets
core_check = check_markers_in_data(CORE_MYELOID, gene_names)
subtype_check = check_markers_in_data(MYELOID_SUBTYPES, gene_names)
contam_check = check_markers_in_data(CONTAMINATION_MARKERS, gene_names)

print("\nCore Myeloid Markers:")
for cat, info in core_check.items():
    print(f"  {cat}: {info['coverage']:.0f}% ({len(info['present'])}/{len(info['present']) + len(info['missing'])})")
    if info['missing']:
        print(f"    Missing: {', '.join(info['missing'])}")

print("\nMyeloid Subtype Markers:")
for cat, info in subtype_check.items():
    if info['coverage'] < 50:
        print(f"  ⚠️  {cat}: {info['coverage']:.0f}% ({len(info['present'])}/{len(info['present']) + len(info['missing'])})")

print("\nContamination Markers:")
for cat, info in contam_check.items():
    print(f"  {cat}: {info['coverage']:.0f}% ({len(info['present'])}/{len(info['present']) + len(info['missing'])})")
    if info['missing']:
        print(f"    Missing: {', '.join(info['missing'])}")

## 4. Global Overview: All Clusters

In [ ]:
# ===== UMAP Overview =====
print("Generating UMAP overviews...")

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Cell type annotation
sc.pl.umap(
    adata,
    color=CELLTYPE_COL,
    ax=axes[0],
    title='Cell Type Annotation',
    legend_loc='right margin',
    legend_fontsize=8,
    show=False
)

# Batch distribution
sc.pl.umap(
    adata,
    color=BATCH_KEY,
    ax=axes[1],
    title='Batch Distribution',
    legend_loc='right margin',
    legend_fontsize=8,
    show=False
)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'overview_umap.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"  ✓ Saved: overview_umap.png")

In [ ]:
# ===== Highlight High-Risk Clusters =====
print("Highlighting high-risk clusters...")

# Create a binary column for high-risk clusters (as strings for stable palette mapping)
adata.obs['is_high_risk'] = pd.Categorical(
    adata.obs[CELLTYPE_COL].isin(HIGH_RISK_CLUSTERS.keys()).map({False: "False", True: "True"}),
    categories=["False", "True"],
    ordered=True,
 )

fig, ax = plt.subplots(figsize=(12, 10))
sc.pl.umap(
    adata,
    color='is_high_risk',
    ax=ax,
    title='High-Risk Clusters (Suspected Doublets/Contaminants)',
    palette={'False': 'lightgray', 'True': 'red'},
    alpha=0.6,
    size=20,
    show=False
)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'high_risk_clusters_umap.png', dpi=300, bbox_inches='tight')
plt.show()

n_high_risk = (adata.obs['is_high_risk'] == "True").sum()
print(f"  ✓ High-risk cells: {n_high_risk:,} ({n_high_risk/adata.n_obs*100:.1f}%)")
print(f"  ✓ Saved: high_risk_clusters_umap.png")

## 5. Core Myeloid Marker Validation

In [ ]:
# ===== Dotplot: Core Myeloid Markers Across All Clusters =====
print("Generating core myeloid marker dotplot...")

# Collect all core myeloid markers that are present
all_core_markers = []
for category, genes in CORE_MYELOID.items():
    present = [g for g in genes if g in gene_names]
    all_core_markers.extend(present)

if all_core_markers:
    # Remove duplicates while preserving order
    all_core_markers = list(dict.fromkeys(all_core_markers))
    
    dotplot = sc.pl.dotplot(
        adata,
        var_names=all_core_markers,
        groupby=CELLTYPE_COL,
        layer='log1p',
        standard_scale='var',
        title='Core Myeloid Markers (ALL should be present in real myeloid cells)',
        show=False,
        return_fig=True,
        figsize=(16, max(12, len(adata.obs[CELLTYPE_COL].unique()) * 0.4)),
    )
    
    dotplot.savefig(OUTPUT_DIR / 'core_myeloid_markers_dotplot.png', dpi=300, bbox_inches='tight')
    dotplot.show()
    
    print(f"  ✓ Saved: core_myeloid_markers_dotplot.png")
    print(f"  ⚠️  CRITICAL: Any cluster with LOW expression of these markers is suspect!")
else:
    print("  ❌ No core myeloid markers found in dataset!")

In [ ]:
# ===== Quantify Core Myeloid Marker Expression =====
print("\nQuantifying core myeloid marker expression per cluster...")

if all_core_markers:
    # Calculate mean expression for each cluster
    core_expr_df = pd.DataFrame()
    
    for celltype in adata.obs[CELLTYPE_COL].unique():
        mask = adata.obs[CELLTYPE_COL] == celltype
        subset = adata[mask]
        
        # Get expression from log1p layer or .raw if available
        if adata.raw is not None:
            # Use .raw for full gene set
            expr_data = subset.raw.to_adata()[:, all_core_markers].X
        else:
            expr_data = subset[:, all_core_markers].X
        
        # Convert to dense if sparse
        if hasattr(expr_data, 'toarray'):
            expr_data = expr_data.toarray()
        
        mean_expr = expr_data.mean(axis=0)
        core_expr_df[celltype] = mean_expr
    
    core_expr_df.index = all_core_markers
    
    # Calculate "myeloid confidence score" (mean of core markers)
    myeloid_score = core_expr_df.mean(axis=0).sort_values(ascending=False)
    
    print("\nMyeloid Confidence Score (mean core marker expression):")
    print("=" * 80)
    for ct, score in myeloid_score.items():
        risk_flag = "⚠️" if ct in HIGH_RISK_CLUSTERS else "✓"
        confidence = "HIGH" if score > 1.0 else "MEDIUM" if score > 0.5 else "LOW"
        print(f"  {risk_flag} {ct}: {score:.3f} ({confidence})")
    print("=" * 80)
    
    # Save to file
    myeloid_score_df = pd.DataFrame({
        'cell_type': myeloid_score.index,
        'myeloid_confidence_score': myeloid_score.values,
        'is_high_risk': [ct in HIGH_RISK_CLUSTERS for ct in myeloid_score.index]
    }).sort_values('myeloid_confidence_score', ascending=False)
    
    myeloid_score_df.to_csv(OUTPUT_DIR / 'myeloid_confidence_scores.csv', index=False)
    print(f"\n  ✓ Saved: myeloid_confidence_scores.csv")

## 6. Contamination Marker Detection

In [ ]:
# ===== Dotplot: Contamination Markers =====
print("Generating contamination marker dotplot...")

# Collect all contamination markers that are present
all_contam_markers = []
for category, genes in CONTAMINATION_MARKERS.items():
    present = [g for g in genes if g in gene_names]
    all_contam_markers.extend(present)

if all_contam_markers:
    # Remove duplicates while preserving order
    all_contam_markers = list(dict.fromkeys(all_contam_markers))
    
    dotplot = sc.pl.dotplot(
        adata,
        var_names=all_contam_markers,
        groupby=CELLTYPE_COL,
        layer='log1p',
        standard_scale='var',
        title='Contamination Markers (should be ABSENT in pure myeloid cells)',
        show=False,
        return_fig=True,
        figsize=(18, max(12, len(adata.obs[CELLTYPE_COL].unique()) * 0.4)),
    )
    
    dotplot.savefig(OUTPUT_DIR / 'contamination_markers_dotplot.png', dpi=300, bbox_inches='tight')
    dotplot.show()
    
    print(f"  ✓ Saved: contamination_markers_dotplot.png")
    print(f"  ⚠️  CRITICAL: High expression = likely doublet or contamination!")
else:
    print("  ⚠️  No contamination markers found in dataset")

In [ ]:
# ===== Quantify Contamination by Category =====
print("\nQuantifying contamination levels by category...")

contamination_summary = pd.DataFrame()

for contam_category, genes in CONTAMINATION_MARKERS.items():
    present_genes = [g for g in genes if g in gene_names]
    
    if not present_genes:
        continue
    
    category_scores = []
    
    for celltype in adata.obs[CELLTYPE_COL].unique():
        mask = adata.obs[CELLTYPE_COL] == celltype
        subset = adata[mask]
        
        # Get expression
        if adata.raw is not None:
            expr_data = subset.raw.to_adata()[:, present_genes].X
        else:
            expr_data = subset[:, present_genes].X
        
        if hasattr(expr_data, 'toarray'):
            expr_data = expr_data.toarray()
        
        mean_expr = expr_data.mean()
        category_scores.append(mean_expr)
    
    contamination_summary[contam_category] = category_scores

contamination_summary.index = adata.obs[CELLTYPE_COL].unique()

# Display high contamination cases
print("\nContamination Detection (values > 0.5 are suspicious):")
print("=" * 80)

for ct in contamination_summary.index:
    high_contam = contamination_summary.loc[ct][contamination_summary.loc[ct] > 0.5]
    
    if len(high_contam) > 0:
        risk_flag = "⚠️" if ct in HIGH_RISK_CLUSTERS else "!"
        print(f"\n{risk_flag} {ct}:")
        for contam_type, score in high_contam.items():
            print(f"    {contam_type}: {score:.3f}")

print("=" * 80)

# Save to file
contamination_summary.to_csv(OUTPUT_DIR / 'contamination_scores_by_category.csv')
print(f"\n  ✓ Saved: contamination_scores_by_category.csv")

## 7. Focused Analysis: High-Risk Clusters

In [ ]:
# ===== Create Detailed Reports for Each High-Risk Cluster =====
print("\nGenerating detailed reports for high-risk clusters...\n")

high_risk_dir = OUTPUT_DIR / 'high_risk_clusters'
high_risk_dir.mkdir(exist_ok=True)

validation_results = []

for cluster_name, suspected_issue in HIGH_RISK_CLUSTERS.items():
    print("=" * 80)
    print(f"Analyzing: {cluster_name}")
    print(f"Suspected issue: {suspected_issue}")
    print("=" * 80)
    
    # Check if cluster exists in data
    if cluster_name not in adata.obs[CELLTYPE_COL].values:
        print(f"  ⚠️  Cluster '{cluster_name}' not found in current data")
        print()
        continue
    
    # Get cluster-specific data
    cluster_mask = adata.obs[CELLTYPE_COL] == cluster_name
    n_cells = cluster_mask.sum()
    
    print(f"  Cells in cluster: {n_cells:,}\n")
    
    # ===== Evidence Collection =====
    evidence = {
        'cluster_name': cluster_name,
        'suspected_issue': suspected_issue,
        'n_cells': n_cells,
    }
    
    # Core myeloid score
    if cluster_name in myeloid_score.index:
        evidence['myeloid_score'] = myeloid_score[cluster_name]
        print(f"  Core myeloid score: {evidence['myeloid_score']:.3f}")
    
    # Contamination scores
    if cluster_name in contamination_summary.index:
        for contam_type in contamination_summary.columns:
            score = contamination_summary.loc[cluster_name, contam_type]
            evidence[f'contam_{contam_type}'] = score
            if score > 0.5:
                print(f"  ⚠️  {contam_type} contamination: {score:.3f}")
    
    # Determine recommendation
    if evidence.get('myeloid_score', 0) < 0.5:
        if any(v > 0.5 for k, v in evidence.items() if k.startswith('contam_')):
            recommendation = "REMOVE - Likely non-myeloid contamination"
            confidence = "HIGH"
        else:
            recommendation = "REVIEW - Low myeloid signature, unclear contamination"
            confidence = "MEDIUM"
    elif any(v > 1.0 for k, v in evidence.items() if k.startswith('contam_')):
        recommendation = "REMOVE - Likely doublet"
        confidence = "HIGH"
    else:
        recommendation = "REVIEW - Moderate evidence for contamination"
        confidence = "MEDIUM"
    
    evidence['recommendation'] = recommendation
    evidence['confidence'] = confidence
    
    print(f"\n  Recommendation: {recommendation}")
    print(f"  Confidence: {confidence}\n")
    
    validation_results.append(evidence)
    
    # ===== Generate Cluster-Specific UMAP =====
    cluster_fig_dir = high_risk_dir / cluster_name.replace('/', '_').replace(' ', '_')
    cluster_fig_dir.mkdir(exist_ok=True)
    
    # Highlight this cluster
    adata.obs['is_current_cluster'] = adata.obs[CELLTYPE_COL] == cluster_name
    
    fig, ax = plt.subplots(figsize=(10, 8))
    sc.pl.umap(
        adata,
        color='is_current_cluster',
        ax=ax,
        title=f'{cluster_name} (n={n_cells:,} cells)',
        palette={True: 'red', False: 'lightgray'},
        size=30,
        show=False
    )
    plt.tight_layout()
    plt.savefig(cluster_fig_dir / 'cluster_location.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"  ✓ Saved: {cluster_fig_dir}/cluster_location.png")
    print()

# Save validation results
validation_df = pd.DataFrame(validation_results)
validation_df.to_csv(OUTPUT_DIR / 'high_risk_cluster_validation_summary.csv', index=False)
print(f"\n✓ Saved comprehensive validation summary: high_risk_cluster_validation_summary.csv")

## 8. Key Marker Expression on UMAP

In [ ]:
# ===== Visualize Critical Markers on UMAP =====
print("Generating UMAP visualizations for key markers...\n")

# Define key markers to visualize
key_markers = {
    'Core_Myeloid': ['LYZ', 'TYROBP', 'FCER1G', 'CSF1R'],
    'Epithelial_Contam': ['EPCAM', 'KRT19', 'BPIFA1', 'SFTPC'],
    'B_Cell_Contam': ['MS4A1', 'CD79A', 'IGKC'],
    'Endothelial_Contam': ['PECAM1', 'VWF', 'EMCN'],
}

for category, markers in key_markers.items():
    present_markers = [m for m in markers if m in gene_names]
    
    if not present_markers:
        print(f"  ⚠️  No {category} markers found in dataset")
        continue
    
    n_markers = len(present_markers)
    ncols = min(2, n_markers)
    nrows = (n_markers + ncols - 1) // ncols
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(10*ncols, 8*nrows))
    if n_markers == 1:
        axes = [axes]
    else:
        axes = axes.flatten() if nrows > 1 else axes
    
    for idx, marker in enumerate(present_markers):
        sc.pl.umap(
            adata,
            color=marker,
            layer='log1p',
            ax=axes[idx],
            title=f'{marker} Expression',
            vmax='p99',
            show=False
        )
    
    # Hide unused subplots
    for idx in range(n_markers, len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    filename = f"{category.lower()}_markers_umap.png"
    plt.savefig(OUTPUT_DIR / filename, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"  ✓ Saved: {filename}")

print("\n  ✓ All marker UMAP visualizations complete")

## 9. Generate Final Re-annotation Recommendations

In [ ]:
# ===== Compile Final Recommendations =====
print("\n" + "=" * 80)
print("FINAL RE-ANNOTATION RECOMMENDATIONS")
print("=" * 80)
print()

if len(validation_results) > 0:
    # Sort by confidence and myeloid score
    validation_df_sorted = validation_df.sort_values(
        ['confidence', 'myeloid_score'],
        ascending=[False, True]
    )
    
    print("High Confidence Recommendations:")
    print("-" * 80)
    high_conf = validation_df_sorted[validation_df_sorted['confidence'] == 'HIGH']
    for _, row in high_conf.iterrows():
        print(f"\n{row['cluster_name']}:")
        print(f"  Current annotation: {row['cluster_name']}")
        print(f"  Suspected issue: {row['suspected_issue']}")
        print(f"  Myeloid score: {row.get('myeloid_score', 'N/A'):.3f}")
        print(f"  Recommendation: {row['recommendation']}")
        
        # Show main contamination source
        contam_cols = [c for c in row.index if c.startswith('contam_')]
        if contam_cols:
            contam_scores = row[contam_cols]
            max_contam = contam_scores.idxmax()
            max_score = contam_scores.max()
            if max_score > 0.5:
                print(f"  Primary contamination: {max_contam.replace('contam_', '')} (score: {max_score:.3f})")
    
    print("\n" + "-" * 80)
    print("\nMedium Confidence Recommendations:")
    print("-" * 80)
    med_conf = validation_df_sorted[validation_df_sorted['confidence'] == 'MEDIUM']
    for _, row in med_conf.iterrows():
        print(f"\n{row['cluster_name']}:")
        print(f"  Recommendation: {row['recommendation']}")
        print(f"  → Requires manual review of marker expression patterns")
    
    print("\n" + "=" * 80)
    
    # Summary statistics
    n_remove = validation_df['recommendation'].str.contains('REMOVE').sum()
    n_review = validation_df['recommendation'].str.contains('REVIEW').sum()
    total_suspect_cells = validation_df['n_cells'].sum()
    
    print(f"\nSummary:")
    print(f"  Clusters flagged for removal: {n_remove}")
    print(f"  Clusters flagged for review: {n_review}")
    print(f"  Total suspect cells: {total_suspect_cells:,} ({total_suspect_cells/adata.n_obs*100:.1f}% of dataset)")
    print("=" * 80)
else:
    print("No high-risk clusters were found in the current dataset.")
    print("=" * 80)

In [ ]:
# ===== Add Validation Flags to Main Object =====
print("\nAdding validation flags to main object...")

# Create validation status column
adata.obs['validation_status'] = 'PASS'

if len(validation_results) > 0:
    for result in validation_results:
        cluster_name = result['cluster_name']
        recommendation = result['recommendation']
        
        mask = adata.obs[CELLTYPE_COL] == cluster_name
        
        if 'REMOVE' in recommendation:
            adata.obs.loc[mask, 'validation_status'] = 'REMOVE'
        elif 'REVIEW' in recommendation:
            adata.obs.loc[mask, 'validation_status'] = 'REVIEW'

# Summary
status_counts = adata.obs['validation_status'].value_counts()
print("\nValidation status distribution:")
for status, count in status_counts.items():
    pct = count / adata.n_obs * 100
    print(f"  {status}: {count:,} cells ({pct:.1f}%)")

# Save updated object
output_h5ad = OUTPUT_DIR / 'adata_myeloid_with_validation.h5ad'
adata.write_h5ad(output_h5ad, compression='gzip', compression_opts=9)
print(f"\n  ✓ Saved updated object: {output_h5ad}")

## 10. Summary & Next Steps

In [ ]:
# ===== Generate Summary Report =====
print("\n" + "=" * 80)
print("ANALYSIS COMPLETE - SUMMARY")
print("=" * 80)
print()
print("Output files generated:")
print("-" * 80)
print(f"  📊 Overview visualizations:")
print(f"     - overview_umap.png")
print(f"     - high_risk_clusters_umap.png")
print()
print(f"  🔬 Marker expression analysis:")
print(f"     - core_myeloid_markers_dotplot.png")
print(f"     - contamination_markers_dotplot.png")
print(f"     - [category]_markers_umap.png (multiple files)")
print()
print(f"  📈 Quantitative assessments:")
print(f"     - myeloid_confidence_scores.csv")
print(f"     - contamination_scores_by_category.csv")
print(f"     - high_risk_cluster_validation_summary.csv")
print()
print(f"  💾 Updated dataset:")
print(f"     - adata_myeloid_with_validation.h5ad")
print()
print(f"  📁 Cluster-specific reports:")
print(f"     - high_risk_clusters/ (detailed per-cluster analysis)")
print("-" * 80)
print()
print("Next steps:")
print("  1. Review high_risk_cluster_validation_summary.csv")
print("  2. Manually inspect figures for 'REMOVE' flagged clusters")
print("  3. For 'REVIEW' clusters, check marker dotplots and UMAPs")
print("  4. Filter out validated doublets/contaminants")
print("  5. Re-run integration and annotation on cleaned dataset")
print()
print("=" * 80)
print(f"\nAll results saved to: {OUTPUT_DIR}")
print("=" * 80)